# WBSNet Model Notebook

This notebook is a **self-contained Kaggle training notebook** for WBSNet.

It does not import from the local `wbsnet/` package. Everything needed to train, validate, evaluate, and export predictions is implemented directly in notebook cells.

What this notebook covers:
- auto-detect the processed `WBSNet_Dataset` uploaded to Kaggle
- build inline dataloaders for the preprocessed directory layout
- define the full WBSNet model, losses, metrics, and visualization utilities
- train with mixed precision, checkpointing, and metric logging
- evaluate the best checkpoint and export qualitative predictions


In [ ]:
# Install any missing lightweight dependencies.
!pip install -q pywavelets


## Runtime Notes

Edit the configuration in the next cell if you want to:
- switch from `kvasir` to `cvc_clinicdb` or `isic2018`
- disable ImageNet pretraining fallback
- change the number of epochs, batch size, or evaluation datasets

The notebook assumes you already uploaded the processed dataset zip produced by the preprocessing notebook.
By default it searches for `WBSNet_Dataset/dataset_info.json` under `/kaggle/input` automatically.


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import re
import time
import warnings
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image, ImageDraw, ImageEnhance
from torch.cuda.amp import GradScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

warnings.filterwarnings("once")


def seed_everything(seed: int, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def autodetect_processed_root(explicit_root: str | None = None) -> Path:
    if explicit_root:
        root = Path(explicit_root).expanduser()
        if not root.exists():
            raise FileNotFoundError(f"Configured dataset root does not exist: {root}")
        return root

    candidates: list[Path] = []
    search_roots = [Path("/kaggle/input"), Path.cwd()]
    for base in search_roots:
        if not base.exists():
            continue

        direct = base / "WBSNet_Dataset"
        if (direct / "dataset_info.json").exists():
            candidates.append(direct)

        for child in sorted(base.iterdir()):
            if not child.is_dir():
                continue
            if child.name == "WBSNet_Dataset" and (child / "dataset_info.json").exists():
                candidates.append(child)
            nested = child / "WBSNet_Dataset"
            if (nested / "dataset_info.json").exists():
                candidates.append(nested)

    if not candidates:
        raise FileNotFoundError(
            "Could not auto-detect WBSNet_Dataset. Set CONFIG['processed_root'] manually to the uploaded dataset path."
        )

    unique_candidates = sorted({str(path.resolve()) for path in candidates})
    return Path(unique_candidates[0])


DATASET_META = {
    "kvasir": {
        "display_name": "Kvasir-SEG",
        "default_extra_evals": [("cvc_colondb", "test")],
    },
    "cvc_clinicdb": {
        "display_name": "CVC-ClinicDB",
        "default_extra_evals": [],
    },
    "cvc_colondb": {
        "display_name": "CVC-ColonDB",
        "default_extra_evals": [],
    },
    "isic2018": {
        "display_name": "ISIC 2018",
        "default_extra_evals": [],
    },
}

default_output_root = "/kaggle/working/wbsnet_runs" if Path("/kaggle/working").exists() else "./wbsnet_runs"

CONFIG = {
    "processed_root": None,
    "train_dataset": "kvasir",
    "eval_specs": None,
    "image_size": [352, 352],
    "batch_size": 4,
    "num_workers": 2,
    "pin_memory": True,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
    "augment": {
        "random_resized_crop": True,
        "random_resized_crop_scale": [0.8, 1.0],
        "random_resized_crop_ratio": [0.9, 1.1],
        "horizontal_flip": True,
        "vertical_flip": True,
        "rotate90": True,
        "color_jitter": True,
    },
    "seed": 42,
    "deterministic": False,
    "output_root": default_output_root,
    "run_name": None,
    "resume_checkpoint": None,
    "encoder_pretrained": True,
    "allow_pretrained_fallback": True,
    "decoder_channels": [256, 128, 64, 32],
    "reduction_ratio": 16,
    "use_wavelet": True,
    "use_lfsa": True,
    "use_hfba": True,
    "boundary_supervision": True,
    "wavelet_type": "haar",
    "epochs": 20,
    "amp": True,
    "grad_accum_steps": 1,
    "encoder_lr": 1e-4,
    "decoder_lr": 1e-3,
    "weight_decay": 1e-4,
    "boundary_loss_weight": 0.5,
    "clip_grad_norm": 1.0,
    "monitor": "dice",
    "monitor_mode": "max",
    "save_every": 5,
    "threshold": 0.5,
    "compute_hd95": True,
    "max_visualizations": 12,
    "contact_sheet_columns": 2,
}

PROCESSED_ROOT = autodetect_processed_root(CONFIG["processed_root"])
DATASET_INFO_PATH = PROCESSED_ROOT / "dataset_info.json"
DATASET_INFO = json.loads(DATASET_INFO_PATH.read_text(encoding="utf-8")) if DATASET_INFO_PATH.exists() else {}
AVAILABLE_DATASETS = sorted(path.name for path in PROCESSED_ROOT.iterdir() if path.is_dir())

if CONFIG["train_dataset"] not in AVAILABLE_DATASETS:
    raise ValueError(
        f"Configured train_dataset='{CONFIG['train_dataset']}' is not present under {PROCESSED_ROOT}. "
        f"Available: {AVAILABLE_DATASETS}"
    )

timestamp = time.strftime("%Y%m%d-%H%M%S")
RUN_NAME = CONFIG["run_name"] or f"wbsnet_{CONFIG['train_dataset']}_seed{CONFIG['seed']}_{timestamp}"
OUTPUT_ROOT = Path(CONFIG["output_root"]).expanduser().resolve()
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
EVAL_DIR = RUN_DIR / "evaluation"
PREDICTION_DIR = RUN_DIR / "predictions"
METRICS_CSV = RUN_DIR / "metrics.csv"

for path in [RUN_DIR, CHECKPOINT_DIR, EVAL_DIR, PREDICTION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

seed_everything(CONFIG["seed"], deterministic=CONFIG["deterministic"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Processed dataset root: {PROCESSED_ROOT}")
print(f"Available datasets: {AVAILABLE_DATASETS}")
print(f"Run directory: {RUN_DIR}")
print(f"Device: {DEVICE}")


## Data Pipeline

The processed dataset layout created by the preprocessing notebook is expected to look like this:

```text
WBSNet_Dataset/
  kvasir/
    train/images, train/masks, train/boundaries
    val/images,   val/masks,   val/boundaries
  cvc_clinicdb/
    train/..., val/...
  cvc_colondb/
    test/...
  isic2018/
    train/..., val/..., test/...
```

The dataloader below:
- reads images, masks, and precomputed boundary maps
- applies the same geometric transforms to all three
- keeps the notebook self-contained


In [ ]:
RESAMPLING = Image.Resampling


def save_json(path: str | Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def natural_sort_key(value: str) -> list[object]:
    return [int(token) if token.isdigit() else token.lower() for token in re.split(r"(\d+)", value)]


def list_file_map(directory: str | Path, suffixes: tuple[str, ...] = (".png", ".jpg", ".jpeg")) -> dict[str, Path]:
    directory = Path(directory)
    files = [path for path in directory.iterdir() if path.is_file() and path.suffix.lower() in suffixes]
    return {path.stem: path for path in sorted(files, key=lambda item: natural_sort_key(item.name))}


def discover_processed_samples(processed_root: str | Path, dataset_name: str, split: str) -> list[dict[str, Any]]:
    split_root = Path(processed_root) / dataset_name / split
    image_dir = split_root / "images"
    mask_dir = split_root / "masks"
    boundary_dir = split_root / "boundaries"

    if not image_dir.exists() or not mask_dir.exists():
        raise FileNotFoundError(f"Missing image or mask directory for {dataset_name}/{split}: {split_root}")
    if not boundary_dir.exists():
        raise FileNotFoundError(f"Missing boundaries directory for {dataset_name}/{split}: {boundary_dir}")

    image_map = list_file_map(image_dir)
    mask_map = list_file_map(mask_dir)
    boundary_map = list_file_map(boundary_dir)

    shared_keys = sorted(set(image_map) & set(mask_map) & set(boundary_map), key=natural_sort_key)
    missing_images = sorted((set(mask_map) | set(boundary_map)) - set(image_map), key=natural_sort_key)
    missing_masks = sorted((set(image_map) | set(boundary_map)) - set(mask_map), key=natural_sort_key)
    missing_boundaries = sorted((set(image_map) | set(mask_map)) - set(boundary_map), key=natural_sort_key)
    if missing_images:
        warnings.warn(f"{dataset_name}/{split}: {len(missing_images)} samples are missing images and will be skipped.")
    if missing_masks:
        warnings.warn(f"{dataset_name}/{split}: {len(missing_masks)} samples are missing masks and will be skipped.")
    if missing_boundaries:
        warnings.warn(f"{dataset_name}/{split}: {len(missing_boundaries)} samples are missing boundaries and will be skipped.")

    return [
        {
            "sample_id": sample_id,
            "image_path": image_map[sample_id],
            "mask_path": mask_map[sample_id],
            "boundary_path": boundary_map[sample_id],
        }
        for sample_id in shared_keys
    ]


@dataclass
class SegmentationTransform:
    image_size: tuple[int, int]
    train: bool
    mean: tuple[float, float, float]
    std: tuple[float, float, float]
    augment: dict[str, Any]

    def __call__(self, image: Image.Image, mask: Image.Image, boundary: Image.Image) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        image = image.convert("RGB")
        mask = mask.convert("L")
        boundary = boundary.convert("L")

        if self.train:
            image, mask, boundary = self._augment(image, mask, boundary)

        target_size = (self.image_size[1], self.image_size[0])
        image = image.resize(target_size, resample=RESAMPLING.BILINEAR)
        mask = mask.resize(target_size, resample=RESAMPLING.NEAREST)
        boundary = boundary.resize(target_size, resample=RESAMPLING.NEAREST)

        image_np = np.asarray(image, dtype=np.float32) / 255.0
        image_np = (image_np - np.array(self.mean, dtype=np.float32)) / np.array(self.std, dtype=np.float32)
        mask_np = (np.asarray(mask, dtype=np.float32) > 127).astype(np.float32)
        boundary_np = (np.asarray(boundary, dtype=np.float32) > 127).astype(np.float32)

        image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).float()
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()
        boundary_tensor = torch.from_numpy(boundary_np).unsqueeze(0).float()
        return image_tensor, mask_tensor, boundary_tensor

    def _augment(
        self,
        image: Image.Image,
        mask: Image.Image,
        boundary: Image.Image,
    ) -> tuple[Image.Image, Image.Image, Image.Image]:
        if self.augment.get("random_resized_crop", False):
            image, mask, boundary = self._random_resized_crop(image, mask, boundary)
        if self.augment.get("horizontal_flip", False) and random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            boundary = boundary.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        if self.augment.get("vertical_flip", False) and random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            mask = mask.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            boundary = boundary.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
        if self.augment.get("rotate90", False) and random.random() < 0.5:
            rotation = random.choice(
                [
                    Image.Transpose.ROTATE_90,
                    Image.Transpose.ROTATE_180,
                    Image.Transpose.ROTATE_270,
                ]
            )
            image = image.transpose(rotation)
            mask = mask.transpose(rotation)
            boundary = boundary.transpose(rotation)
        if self.augment.get("color_jitter", False):
            image = self._color_jitter(image)
        return image, mask, boundary

    @staticmethod
    def _color_jitter(image: Image.Image) -> Image.Image:
        for enhancer_cls in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
            factor = random.uniform(0.9, 1.1)
            image = enhancer_cls(image).enhance(factor)
        return image

    def _random_resized_crop(
        self,
        image: Image.Image,
        mask: Image.Image,
        boundary: Image.Image,
    ) -> tuple[Image.Image, Image.Image, Image.Image]:
        width, height = image.size
        area = width * height
        scale = tuple(self.augment.get("random_resized_crop_scale", [0.8, 1.0]))
        ratio = tuple(self.augment.get("random_resized_crop_ratio", [0.9, 1.1]))
        log_ratio = (math.log(ratio[0]), math.log(ratio[1]))

        for _ in range(10):
            target_area = random.uniform(scale[0], scale[1]) * area
            aspect_ratio = math.exp(random.uniform(*log_ratio))
            crop_width = int(round(math.sqrt(target_area * aspect_ratio)))
            crop_height = int(round(math.sqrt(target_area / aspect_ratio)))
            if 0 < crop_width <= width and 0 < crop_height <= height:
                left = random.randint(0, width - crop_width)
                top = random.randint(0, height - crop_height)
                box = (left, top, left + crop_width, top + crop_height)
                return image.crop(box), mask.crop(box), boundary.crop(box)

        min_side = min(width, height)
        left = (width - min_side) // 2
        top = (height - min_side) // 2
        box = (left, top, left + min_side, top + min_side)
        return image.crop(box), mask.crop(box), boundary.crop(box)


class ProcessedWBSNetDataset(Dataset):
    def __init__(
        self,
        processed_root: str | Path,
        dataset_name: str,
        split: str,
        train: bool,
        config: dict[str, Any],
    ) -> None:
        self.processed_root = Path(processed_root)
        self.dataset_name = dataset_name
        self.split = split
        self.records = discover_processed_samples(self.processed_root, dataset_name, split)
        self.transform = SegmentationTransform(
            image_size=tuple(config["image_size"]),
            train=train,
            mean=tuple(config["normalize_mean"]),
            std=tuple(config["normalize_std"]),
            augment=config["augment"],
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        record = self.records[index]
        with Image.open(record["image_path"]) as image:
            with Image.open(record["mask_path"]) as mask:
                with Image.open(record["boundary_path"]) as boundary:
                    image_tensor, mask_tensor, boundary_tensor = self.transform(image, mask, boundary)
        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "boundary": boundary_tensor,
            "sample_id": record["sample_id"],
            "image_path": str(record["image_path"]),
            "mask_path": str(record["mask_path"]),
            "boundary_path": str(record["boundary_path"]),
        }


def build_dataloader(dataset: Dataset, batch_size: int, shuffle: bool, config: dict[str, Any]) -> DataLoader:
    num_workers = int(config["num_workers"])
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=bool(config["pin_memory"] and torch.cuda.is_available()),
        persistent_workers=bool(num_workers > 0),
        drop_last=shuffle,
    )


def denormalize_image_tensor(image: torch.Tensor, mean: list[float], std: list[float]) -> np.ndarray:
    image_np = image.detach().cpu().numpy().transpose(1, 2, 0)
    mean_np = np.array(mean, dtype=np.float32)
    std_np = np.array(std, dtype=np.float32)
    image_np = (image_np * std_np) + mean_np
    image_np = np.clip(image_np * 255.0, 0, 255).astype(np.uint8)
    return image_np


def tensor_to_mask_uint8(mask: torch.Tensor) -> np.ndarray:
    mask_np = mask.detach().cpu().numpy()
    if mask_np.ndim == 3:
        mask_np = mask_np[0]
    return (mask_np > 0.5).astype(np.uint8) * 255


def show_samples(dataset: Dataset, count: int, title: str) -> None:
    count = min(count, len(dataset))
    if count == 0:
        print(f"No samples available for {title}.")
        return
    indices = list(range(count))
    fig, axes = plt.subplots(count, 3, figsize=(11, 4 * count))
    if count == 1:
        axes = np.expand_dims(axes, axis=0)
    for row, idx in enumerate(indices):
        sample = dataset[idx]
        image_np = denormalize_image_tensor(sample["image"], CONFIG["normalize_mean"], CONFIG["normalize_std"])
        mask_np = tensor_to_mask_uint8(sample["mask"])
        boundary_np = tensor_to_mask_uint8(sample["boundary"])
        axes[row, 0].imshow(image_np)
        axes[row, 0].set_title(f"{title} | {sample['sample_id']} | image")
        axes[row, 1].imshow(mask_np, cmap="gray", vmin=0, vmax=255)
        axes[row, 1].set_title("mask")
        axes[row, 2].imshow(boundary_np, cmap="gray", vmin=0, vmax=255)
        axes[row, 2].set_title("boundary")
        for axis in axes[row]:
            axis.axis("off")
    plt.tight_layout()
    plt.show()


def infer_eval_specs(train_dataset_name: str, explicit_specs: list[dict[str, str]] | None = None) -> list[dict[str, str]]:
    if explicit_specs:
        return explicit_specs
    specs = [{"dataset": train_dataset_name, "split": "val"}]
    if (PROCESSED_ROOT / train_dataset_name / "test").exists():
        specs.append({"dataset": train_dataset_name, "split": "test"})
    for dataset_name, split_name in DATASET_META.get(train_dataset_name, {}).get("default_extra_evals", []):
        if (PROCESSED_ROOT / dataset_name / split_name).exists():
            specs.append({"dataset": dataset_name, "split": split_name})
    return specs


In [ ]:
TRAIN_DATASET = ProcessedWBSNetDataset(
    processed_root=PROCESSED_ROOT,
    dataset_name=CONFIG["train_dataset"],
    split="train",
    train=True,
    config=CONFIG,
)
VAL_DATASET = ProcessedWBSNetDataset(
    processed_root=PROCESSED_ROOT,
    dataset_name=CONFIG["train_dataset"],
    split="val",
    train=False,
    config=CONFIG,
)
PREVIEW_DATASET = ProcessedWBSNetDataset(
    processed_root=PROCESSED_ROOT,
    dataset_name=CONFIG["train_dataset"],
    split="val",
    train=False,
    config=CONFIG,
)

TRAIN_LOADER = build_dataloader(TRAIN_DATASET, batch_size=CONFIG["batch_size"], shuffle=True, config=CONFIG)
VAL_LOADER = build_dataloader(VAL_DATASET, batch_size=CONFIG["batch_size"], shuffle=False, config=CONFIG)
EVAL_SPECS = infer_eval_specs(CONFIG["train_dataset"], CONFIG["eval_specs"])

print(f"Training dataset: {CONFIG['train_dataset']} ({DATASET_META[CONFIG['train_dataset']]['display_name']})")
print(f"Train samples: {len(TRAIN_DATASET)}")
print(f"Val samples:   {len(VAL_DATASET)}")
print("Evaluation specs:")
for spec in EVAL_SPECS:
    print(f"  - {spec['dataset']} / {spec['split']}")

save_json(
    RUN_DIR / "resolved_config.json",
    {
        **CONFIG,
        "processed_root": str(PROCESSED_ROOT),
        "run_name": RUN_NAME,
        "device": str(DEVICE),
        "eval_specs": EVAL_SPECS,
    },
)

show_samples(PREVIEW_DATASET, count=3, title=f"{CONFIG['train_dataset']} validation preview")


## WBSNet Architecture

The next two cells implement the entire model inline:
- low-frequency semantic attention (LFSA)
- high-frequency boundary attention (HFBA)
- wavelet decomposition and reconstruction
- ResNet-34-style encoder
- U-Net-style decoder with WBS skip refinement


In [ ]:
class LFSA(nn.Module):
    def __init__(self, channels: int, reduction_ratio: int = 16) -> None:
        super().__init__()
        reduced = max(channels // reduction_ratio, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, reduced, kernel_size=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced, channels, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.fc(self.pool(x))


class HFBA(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        merged_channels = channels * 3
        self.depthwise = nn.Sequential(
            nn.Conv2d(merged_channels, merged_channels, kernel_size=3, padding=1, groups=merged_channels, bias=False),
            nn.BatchNorm2d(merged_channels),
            nn.ReLU(inplace=True),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(merged_channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.boundary_head = nn.Conv2d(channels, 1, kernel_size=1)

    def forward(self, lh: torch.Tensor, hl: torch.Tensor, hh: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        merged = torch.cat([lh, hl, hh], dim=1)
        reduced = self.pointwise(self.depthwise(merged))
        boundary_logits = self.boundary_head(reduced)
        attended = reduced * torch.sigmoid(boundary_logits)
        return attended, boundary_logits


def _haar_filters() -> tuple[list[float], list[float], list[float], list[float]]:
    scale = 2 ** -0.5
    dec_lo = [scale, scale]
    dec_hi = [-scale, scale]
    rec_lo = [scale, scale]
    rec_hi = [scale, -scale]
    return dec_lo, dec_hi, rec_lo, rec_hi


@lru_cache(maxsize=8)
def _wavelet_filters(wavelet_type: str) -> tuple[list[float], list[float], list[float], list[float]]:
    wavelet_type = str(wavelet_type).lower()
    if wavelet_type == "haar":
        return _haar_filters()
    try:
        import pywt

        wavelet = pywt.Wavelet(wavelet_type)
        return list(wavelet.dec_lo), list(wavelet.dec_hi), list(wavelet.rec_lo), list(wavelet.rec_hi)
    except Exception as exc:
        raise NotImplementedError(
            f"Wavelet '{wavelet_type}' requires pywavelets or an explicit implementation."
        ) from exc


def _stack_filters(lo: list[float], hi: list[float], device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    lo_tensor = torch.tensor(lo, device=device, dtype=dtype)
    hi_tensor = torch.tensor(hi, device=device, dtype=dtype)
    ll = torch.outer(lo_tensor, lo_tensor)
    lh = torch.outer(lo_tensor, hi_tensor)
    hl = torch.outer(hi_tensor, lo_tensor)
    hh = torch.outer(hi_tensor, hi_tensor)
    return torch.stack([ll, lh, hl, hh], dim=0).unsqueeze(1)


class WaveletTransform2d(nn.Module):
    def __init__(self, wavelet_type: str = "haar") -> None:
        super().__init__()
        self.wavelet_type = wavelet_type

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        batch_size, channels, height, width = x.shape
        if height % 2 != 0 or width % 2 != 0:
            raise ValueError(f"Wavelet input must have even spatial size, got {x.shape}")
        dec_lo, dec_hi, _, _ = _wavelet_filters(self.wavelet_type)
        base_filters = _stack_filters(dec_lo, dec_hi, x.device, x.dtype)
        padding = max(len(dec_lo) // 2 - 1, 0)
        weight = base_filters.repeat(channels, 1, 1, 1)
        transformed = F.conv2d(x, weight, stride=2, padding=padding, groups=channels)
        transformed = transformed.view(batch_size, channels, 4, transformed.shape[-2], transformed.shape[-1])
        return transformed[:, :, 0], transformed[:, :, 1], transformed[:, :, 2], transformed[:, :, 3]


class InverseWaveletTransform2d(nn.Module):
    def __init__(self, wavelet_type: str = "haar") -> None:
        super().__init__()
        self.wavelet_type = wavelet_type

    def forward(
        self,
        ll: torch.Tensor,
        lh: torch.Tensor,
        hl: torch.Tensor,
        hh: torch.Tensor,
    ) -> torch.Tensor:
        _, channels, _, _ = ll.shape
        _, _, rec_lo, rec_hi = _wavelet_filters(self.wavelet_type)
        base_filters = _stack_filters(rec_lo, rec_hi, ll.device, ll.dtype)
        padding = max(len(rec_lo) // 2 - 1, 0)
        packed = torch.stack([ll, lh, hl, hh], dim=2).reshape(ll.shape[0], channels * 4, ll.shape[-2], ll.shape[-1])
        weight = base_filters.repeat(channels, 1, 1, 1)
        return F.conv_transpose2d(packed, weight, stride=2, padding=padding, groups=channels)


class RawAttentionSkip(nn.Module):
    def __init__(self, channels: int, reduction_ratio: int = 16, use_lfsa: bool = True, use_hfba: bool = True) -> None:
        super().__init__()
        self.channel_attention = LFSA(channels, reduction_ratio) if use_lfsa else nn.Identity()
        if use_hfba:
            self.spatial = nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(channels, 1, kernel_size=1),
            )
        else:
            self.spatial = None

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:
        out = self.channel_attention(x)
        if self.spatial is None:
            return out, None
        boundary_logits = self.spatial(out)
        out = out * torch.sigmoid(boundary_logits)
        return out, boundary_logits


class WBSModule(nn.Module):
    def __init__(
        self,
        channels: int,
        reduction_ratio: int = 16,
        use_wavelet: bool = True,
        use_lfsa: bool = True,
        use_hfba: bool = True,
        boundary_supervision: bool = True,
        wavelet_type: str = "haar",
    ) -> None:
        super().__init__()
        self.use_wavelet = use_wavelet
        self.boundary_supervision = boundary_supervision
        self.dwt = WaveletTransform2d(wavelet_type=wavelet_type)
        self.idwt = InverseWaveletTransform2d(wavelet_type=wavelet_type)
        self.lfsa = LFSA(channels, reduction_ratio) if use_lfsa else nn.Identity()
        self.hfba = HFBA(channels) if use_hfba else None
        self.raw_attention = RawAttentionSkip(channels, reduction_ratio, use_lfsa, use_hfba)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:
        if not self.use_wavelet:
            refined, boundary_logits = self.raw_attention(x)
            if not self.boundary_supervision:
                boundary_logits = None
            return refined, boundary_logits

        ll, lh, hl, hh = self.dwt(x)
        ll = self.lfsa(ll)

        if self.hfba is not None:
            hf, boundary_logits = self.hfba(lh, hl, hh)
            lh = hf
            hl = hf
            hh = hf
        else:
            boundary_logits = None

        refined = self.idwt(ll, lh, hl, hh)
        if not self.boundary_supervision:
            boundary_logits = None
        return refined, boundary_logits


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int) -> None:
        super().__init__()
        self.conv = ConvBlock(in_channels + skip_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.downsample = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class ResNetEncoder(nn.Module):
    def __init__(self, in_channels: int = 3) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 64, blocks=3, stride=1)
        self.layer2 = self._make_layer(64, 128, blocks=4, stride=2)
        self.layer3 = self._make_layer(128, 256, blocks=6, stride=2)
        self.layer4 = self._make_layer(256, 512, blocks=3, stride=2)

    @staticmethod
    def _make_layer(in_channels: int, out_channels: int, blocks: int, stride: int) -> nn.Sequential:
        layers = [BasicBlock(in_channels, out_channels, stride=stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        stem = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(stem)
        layer1 = self.layer1(x)
        layer2 = self.layer2(layer1)
        layer3 = self.layer3(layer2)
        layer4 = self.layer4(layer3)
        return stem, layer1, layer2, layer3, layer4

    def load_imagenet_pretrained(self, fail_silently: bool = True) -> None:
        try:
            from torchvision.models import ResNet34_Weights, resnet34

            state = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1).state_dict()
            current = self.state_dict()
            compatible = {
                key: value
                for key, value in state.items()
                if key in current and tuple(value.shape) == tuple(current[key].shape)
            }
            self.load_state_dict(compatible, strict=False)
            print(f"Loaded {len(compatible)} ImageNet-pretrained encoder tensors.")
        except Exception as exc:
            message = (
                "ImageNet pretrained weights could not be loaded. "
                "Continuing with randomly initialized encoder weights."
            )
            if fail_silently:
                warnings.warn(f"{message} Original error: {exc}")
            else:
                raise RuntimeError(message) from exc


class WBSNet(nn.Module):
    def __init__(self, config: dict[str, Any]) -> None:
        super().__init__()
        decoder_channels = config.get("decoder_channels", [256, 128, 64, 32])
        reduction_ratio = int(config.get("reduction_ratio", 16))

        self.encoder = ResNetEncoder(in_channels=3)
        self.skip_modules = nn.ModuleList(
            [
                WBSModule(
                    64,
                    reduction_ratio=reduction_ratio,
                    use_wavelet=bool(config.get("use_wavelet", True)),
                    use_lfsa=bool(config.get("use_lfsa", True)),
                    use_hfba=bool(config.get("use_hfba", True)),
                    boundary_supervision=bool(config.get("boundary_supervision", True)),
                    wavelet_type=str(config.get("wavelet_type", "haar")),
                ),
                WBSModule(
                    64,
                    reduction_ratio=reduction_ratio,
                    use_wavelet=bool(config.get("use_wavelet", True)),
                    use_lfsa=bool(config.get("use_lfsa", True)),
                    use_hfba=bool(config.get("use_hfba", True)),
                    boundary_supervision=bool(config.get("boundary_supervision", True)),
                    wavelet_type=str(config.get("wavelet_type", "haar")),
                ),
                WBSModule(
                    128,
                    reduction_ratio=reduction_ratio,
                    use_wavelet=bool(config.get("use_wavelet", True)),
                    use_lfsa=bool(config.get("use_lfsa", True)),
                    use_hfba=bool(config.get("use_hfba", True)),
                    boundary_supervision=bool(config.get("boundary_supervision", True)),
                    wavelet_type=str(config.get("wavelet_type", "haar")),
                ),
                WBSModule(
                    256,
                    reduction_ratio=reduction_ratio,
                    use_wavelet=bool(config.get("use_wavelet", True)),
                    use_lfsa=bool(config.get("use_lfsa", True)),
                    use_hfba=bool(config.get("use_hfba", True)),
                    boundary_supervision=bool(config.get("boundary_supervision", True)),
                    wavelet_type=str(config.get("wavelet_type", "haar")),
                ),
            ]
        )

        self.dec4 = DecoderBlock(512, 256, decoder_channels[0])
        self.dec3 = DecoderBlock(decoder_channels[0], 128, decoder_channels[1])
        self.dec2 = DecoderBlock(decoder_channels[1], 64, decoder_channels[2])
        self.dec1 = DecoderBlock(decoder_channels[2], 64, decoder_channels[3])
        self.head = nn.Conv2d(decoder_channels[3], 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> dict[str, Any]:
        stem, layer1, layer2, layer3, bottleneck = self.encoder(x)
        skip1, bnd1 = self.skip_modules[0](stem)
        skip2, bnd2 = self.skip_modules[1](layer1)
        skip3, bnd3 = self.skip_modules[2](layer2)
        skip4, bnd4 = self.skip_modules[3](layer3)

        dec4 = self.dec4(bottleneck, skip4)
        dec3 = self.dec3(dec4, skip3)
        dec2 = self.dec2(dec3, skip2)
        dec1 = self.dec1(dec2, skip1)
        logits = self.head(F.interpolate(dec1, scale_factor=2, mode="bilinear", align_corners=False))
        boundary_logits = [item for item in [bnd1, bnd2, bnd3, bnd4] if item is not None]
        return {"logits": logits, "boundary_logits": boundary_logits}


def count_parameters(model: nn.Module) -> tuple[int, int]:
    total = sum(param.numel() for param in model.parameters())
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    return total, trainable


## Losses, Metrics, Training Utilities

The next two cells implement:
- segmentation and boundary losses
- Dice, IoU, precision, recall, specificity, and HD95
- checkpointing and prediction export helpers
- the training and evaluation loops used by the notebook


In [ ]:
def dice_loss_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, targets.ndim))
    intersection = (probs * targets).sum(dim=dims)
    union = probs.sum(dim=dims) + targets.sum(dim=dims)
    dice = (2.0 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()


def segmentation_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dice = dice_loss_from_logits(logits, targets)
    return bce + dice


def boundary_loss(boundary_logits: list[torch.Tensor], boundary_targets: torch.Tensor) -> torch.Tensor:
    if not boundary_logits:
        return boundary_targets.new_tensor(0.0)
    losses = []
    for logits in boundary_logits:
        resized_targets = F.interpolate(boundary_targets, size=logits.shape[-2:], mode="nearest")
        losses.append(F.binary_cross_entropy_with_logits(logits, resized_targets))
    return torch.stack(losses).mean()


def total_loss(
    model_output: dict[str, Any],
    masks: torch.Tensor,
    boundaries: torch.Tensor,
    boundary_weight: float,
) -> tuple[torch.Tensor, dict[str, float]]:
    seg = segmentation_loss(model_output["logits"], masks)
    bnd = boundary_loss(model_output.get("boundary_logits", []), boundaries)
    total = seg + boundary_weight * bnd
    return total, {
        "segmentation_loss": float(seg.detach().item()),
        "boundary_loss": float(bnd.detach().item()),
        "total_loss": float(total.detach().item()),
    }


def safe_divide(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator > 0 else 0.0


def mask_boundary(mask: np.ndarray) -> np.ndarray:
    padded = np.pad(mask.astype(np.uint8), 1, mode="constant")
    neighbors = [
        padded[1:-1, :-2],
        padded[1:-1, 2:],
        padded[:-2, 1:-1],
        padded[2:, 1:-1],
    ]
    same = np.logical_and.reduce([mask == neighbor for neighbor in neighbors])
    return np.logical_and(mask.astype(bool), np.logical_not(same))


def pairwise_min_distances(points_a: np.ndarray, points_b: np.ndarray, chunk_size: int = 1024) -> np.ndarray:
    if len(points_a) == 0 or len(points_b) == 0:
        return np.array([], dtype=np.float32)
    mins: list[np.ndarray] = []
    for start in range(0, len(points_a), chunk_size):
        chunk = points_a[start : start + chunk_size]
        distances = np.sqrt(((chunk[:, None, :] - points_b[None, :, :]) ** 2).sum(axis=2))
        mins.append(distances.min(axis=1))
    return np.concatenate(mins, axis=0)


def hd95_score(pred_mask: np.ndarray, target_mask: np.ndarray) -> float:
    pred_mask = pred_mask.astype(bool)
    target_mask = target_mask.astype(bool)
    if pred_mask.sum() == 0 and target_mask.sum() == 0:
        return 0.0
    if pred_mask.sum() == 0 or target_mask.sum() == 0:
        return float(np.hypot(*pred_mask.shape))

    pred_boundary = np.argwhere(mask_boundary(pred_mask))
    target_boundary = np.argwhere(mask_boundary(target_mask))
    if len(pred_boundary) == 0 or len(target_boundary) == 0:
        return float(np.hypot(*pred_mask.shape))

    pred_to_target = pairwise_min_distances(pred_boundary, target_boundary)
    target_to_pred = pairwise_min_distances(target_boundary, pred_boundary)
    distances = np.concatenate([pred_to_target, target_to_pred], axis=0)
    return float(np.percentile(distances, 95))


@dataclass
class BinarySegmentationMeter:
    threshold: float = 0.5
    compute_hd95: bool = True
    counts: dict[str, float] = field(
        default_factory=lambda: {
            "tp": 0.0,
            "fp": 0.0,
            "fn": 0.0,
            "tn": 0.0,
            "loss": 0.0,
            "num_batches": 0.0,
        }
    )
    hd95_values: list[float] = field(default_factory=list)

    def update(self, logits: torch.Tensor, targets: torch.Tensor, loss_value: float) -> None:
        probs = torch.sigmoid(logits)
        preds = (probs >= self.threshold).float()
        self.counts["tp"] += float(((preds == 1) & (targets == 1)).sum().item())
        self.counts["fp"] += float(((preds == 1) & (targets == 0)).sum().item())
        self.counts["fn"] += float(((preds == 0) & (targets == 1)).sum().item())
        self.counts["tn"] += float(((preds == 0) & (targets == 0)).sum().item())
        self.counts["loss"] += float(loss_value)
        self.counts["num_batches"] += 1.0

        if self.compute_hd95:
            pred_np = preds.detach().cpu().numpy()
            target_np = targets.detach().cpu().numpy()
            for pred_sample, target_sample in zip(pred_np, target_np):
                self.hd95_values.append(hd95_score(pred_sample[0], target_sample[0]))

    def compute(self) -> dict[str, float]:
        tp = self.counts["tp"]
        fp = self.counts["fp"]
        fn = self.counts["fn"]
        tn = self.counts["tn"]
        metrics = {
            "loss": safe_divide(self.counts["loss"], self.counts["num_batches"]),
            "dice": safe_divide(2.0 * tp, 2.0 * tp + fp + fn),
            "iou": safe_divide(tp, tp + fp + fn),
            "precision": safe_divide(tp, tp + fp),
            "recall": safe_divide(tp, tp + fn),
            "accuracy": safe_divide(tp + tn, tp + tn + fp + fn),
            "specificity": safe_divide(tn, tn + fp),
        }
        if self.hd95_values:
            metrics["hd95"] = float(np.mean(self.hd95_values))
        return metrics


def mean_scalar_dict(totals: dict[str, float], count: int) -> dict[str, float]:
    if count <= 0:
        return {key: 0.0 for key in totals}
    return {key: float(value / count) for key, value in totals.items()}


def format_metrics(metrics: dict[str, float], keys: list[str]) -> str:
    return " | ".join(f"{key}: {metrics[key]:.4f}" for key in keys if key in metrics)


In [ ]:
def make_overlay(image_np: np.ndarray, target_np: np.ndarray, pred_np: np.ndarray) -> np.ndarray:
    overlay = image_np.copy()
    overlay[..., 0] = np.where(pred_np > 0, 255, overlay[..., 0])
    overlay[..., 1] = np.where(target_np > 0, 255, overlay[..., 1])
    return overlay


def tile_with_title(tile: np.ndarray, title: str, title_height: int = 28) -> np.ndarray:
    canvas = np.full((tile.shape[0] + title_height, tile.shape[1], 3), 255, dtype=np.uint8)
    canvas[title_height:, :, :] = tile
    image = Image.fromarray(canvas)
    draw = ImageDraw.Draw(image)
    draw.text((8, 6), title, fill=(0, 0, 0))
    return np.asarray(image)


def find_focus_box(target_np: np.ndarray, pred_np: np.ndarray, min_size: int = 96, padding: int = 24) -> tuple[int, int, int, int]:
    target_mask = target_np > 0
    pred_mask = pred_np > 0
    focus = np.logical_xor(target_mask, pred_mask)
    if not focus.any():
        focus = np.logical_or(target_mask, pred_mask)

    height, width = target_np.shape[:2]
    if not focus.any():
        crop = min(min_size, height, width)
        top = max((height - crop) // 2, 0)
        left = max((width - crop) // 2, 0)
        return left, top, min(left + crop, width), min(top + crop, height)

    ys, xs = np.where(focus)
    left = max(int(xs.min()) - padding, 0)
    right = min(int(xs.max()) + padding + 1, width)
    top = max(int(ys.min()) - padding, 0)
    bottom = min(int(ys.max()) + padding + 1, height)

    box_width = right - left
    box_height = bottom - top
    target_size = max(min_size, box_width, box_height)
    center_x = (left + right) // 2
    center_y = (top + bottom) // 2
    half = target_size // 2
    left = max(center_x - half, 0)
    top = max(center_y - half, 0)
    right = min(left + target_size, width)
    bottom = min(top + target_size, height)
    left = max(right - target_size, 0)
    top = max(bottom - target_size, 0)
    return left, top, right, bottom


def draw_focus_box(tile: np.ndarray, box: tuple[int, int, int, int], color: tuple[int, int, int] = (255, 0, 0)) -> np.ndarray:
    image = Image.fromarray(tile)
    draw = ImageDraw.Draw(image)
    left, top, right, bottom = box
    draw.rectangle([left, top, right - 1, bottom - 1], outline=color, width=3)
    return np.asarray(image)


def create_prediction_visuals(
    image: torch.Tensor,
    target_mask: torch.Tensor,
    pred_mask: torch.Tensor,
    mean: list[float],
    std: list[float],
    zoom_size: int = 256,
) -> dict[str, np.ndarray | tuple[int, int, int, int]]:
    image_np = denormalize_image_tensor(image, mean, std)
    target_np = tensor_to_mask_uint8(target_mask)
    pred_np = tensor_to_mask_uint8(pred_mask)
    overlay_np = make_overlay(image_np, target_np, pred_np)

    focus_box = find_focus_box(target_np, pred_np)
    left, top, right, bottom = focus_box
    image_box = draw_focus_box(image_np, focus_box)
    target_box = draw_focus_box(np.repeat(target_np[..., None], 3, axis=2), focus_box)
    pred_box = draw_focus_box(np.repeat(pred_np[..., None], 3, axis=2), focus_box)
    overlay_box = draw_focus_box(overlay_np, focus_box)
    zoom_crop = overlay_np[top:bottom, left:right]
    zoom_panel = np.asarray(Image.fromarray(zoom_crop).resize((zoom_size, zoom_size), resample=RESAMPLING.NEAREST))

    paper_panel = np.concatenate(
        [
            tile_with_title(image_box, "Image"),
            tile_with_title(target_box, "Ground Truth"),
            tile_with_title(pred_box, "Prediction"),
            tile_with_title(overlay_box, "Overlay"),
            tile_with_title(zoom_panel, "Boundary Zoom"),
        ],
        axis=1,
    )
    return {
        "image": image_np,
        "target": target_np,
        "prediction": pred_np,
        "overlay": overlay_np,
        "focus_box": focus_box,
        "paper_panel": paper_panel,
    }


def save_prediction_triplet(
    save_dir: str | Path,
    sample_id: str,
    image: torch.Tensor,
    target_mask: torch.Tensor,
    pred_mask: torch.Tensor,
    mean: list[float],
    std: list[float],
) -> dict[str, str]:
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    visuals = create_prediction_visuals(image, target_mask, pred_mask, mean, std)

    image_path = save_dir / f"{sample_id}_image.png"
    target_path = save_dir / f"{sample_id}_target.png"
    pred_path = save_dir / f"{sample_id}_prediction.png"
    overlay_path = save_dir / f"{sample_id}_overlay.png"
    panel_path = save_dir / f"{sample_id}_paper_panel.png"

    Image.fromarray(visuals["image"]).save(image_path)
    Image.fromarray(visuals["target"]).save(target_path)
    Image.fromarray(visuals["prediction"]).save(pred_path)
    Image.fromarray(visuals["overlay"]).save(overlay_path)
    Image.fromarray(visuals["paper_panel"]).save(panel_path)

    return {
        "image": str(image_path),
        "target": str(target_path),
        "prediction": str(pred_path),
        "overlay": str(overlay_path),
        "paper_panel": str(panel_path),
    }


def save_contact_sheet(
    panel_paths: list[str | Path],
    output_path: str | Path,
    columns: int = 2,
    panel_spacing: int = 20,
    margin: int = 20,
) -> str:
    if not panel_paths:
        raise ValueError("At least one paper panel is required to build a contact sheet.")
    opened_panels = [Image.open(path).convert("RGB") for path in panel_paths]
    try:
        max_width = max(panel.width for panel in opened_panels)
        max_height = max(panel.height for panel in opened_panels)
        columns = max(1, columns)
        rows = (len(opened_panels) + columns - 1) // columns
        canvas_width = (columns * max_width) + ((columns - 1) * panel_spacing) + (2 * margin)
        canvas_height = (rows * max_height) + ((rows - 1) * panel_spacing) + (2 * margin)
        canvas = Image.new("RGB", (canvas_width, canvas_height), (255, 255, 255))
        for index, panel in enumerate(opened_panels):
            row = index // columns
            column = index % columns
            x = margin + column * (max_width + panel_spacing)
            y = margin + row * (max_height + panel_spacing)
            canvas.paste(panel, (x, y))
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        canvas.save(output_path)
        return str(output_path)
    finally:
        for panel in opened_panels:
            panel.close()


def build_optimizer(model: nn.Module, config: dict[str, Any]) -> tuple[torch.optim.Optimizer, torch.optim.lr_scheduler._LRScheduler]:
    encoder_params = list(model.encoder.parameters())
    encoder_param_ids = {id(param) for param in encoder_params}
    decoder_params = [param for param in model.parameters() if id(param) not in encoder_param_ids]
    optimizer = torch.optim.AdamW(
        [
            {"params": encoder_params, "lr": float(config["encoder_lr"])},
            {"params": decoder_params, "lr": float(config["decoder_lr"])},
        ],
        weight_decay=float(config["weight_decay"]),
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(config["epochs"]))
    return optimizer, scheduler


def save_checkpoint(
    path: str | Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    scaler: GradScaler,
    epoch: int,
    best_metric: float,
    config: dict[str, Any],
) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config,
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
        },
        path,
    )


def load_checkpoint(
    path: str | Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
    scheduler: torch.optim.lr_scheduler._LRScheduler | None = None,
    scaler: GradScaler | None = None,
    map_location: str | torch.device = "cpu",
) -> dict[str, Any]:
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    if optimizer is not None and "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if scheduler is not None and "scheduler" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler"])
    if scaler is not None and "scaler" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler"])
    return checkpoint


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None,
    scaler: GradScaler,
    device: torch.device,
    config: dict[str, Any],
    training: bool,
) -> dict[str, float]:
    amp_enabled = bool(config["amp"] and device.type == "cuda")
    grad_accum_steps = int(config["grad_accum_steps"])
    clip_grad_norm = float(config["clip_grad_norm"])
    meter = BinarySegmentationMeter(
        threshold=float(config["threshold"]),
        compute_hd95=bool(config["compute_hd95"]),
    )
    loss_totals = {"segmentation_loss": 0.0, "boundary_loss": 0.0, "total_loss": 0.0}
    loss_steps = 0

    model.train(training)
    if training and optimizer is not None:
        optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc="train" if training else "eval", leave=False)
    for step, batch in enumerate(progress, start=1):
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        boundaries = batch["boundary"].to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, enabled=amp_enabled):
                output = model(images)
                loss, loss_parts = total_loss(
                    output,
                    masks,
                    boundaries,
                    boundary_weight=float(config["boundary_loss_weight"]),
                )
                scaled_loss = loss / grad_accum_steps

            if training and optimizer is not None:
                scaler.scale(scaled_loss).backward()
                if step % grad_accum_steps == 0:
                    if clip_grad_norm > 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

        meter.update(output["logits"].detach(), masks.detach(), float(loss.detach().item()))
        for key in loss_totals:
            loss_totals[key] += float(loss_parts[key])
        loss_steps += 1
        progress.set_postfix(loss=f"{loss.item():.4f}", dice=f"{meter.compute().get('dice', 0.0):.4f}")

    if training and optimizer is not None and len(loader) % grad_accum_steps != 0:
        if clip_grad_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    metrics = meter.compute()
    metrics.update(mean_scalar_dict(loss_totals, loss_steps))
    return metrics


@torch.no_grad()
def evaluate_and_export(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    config: dict[str, Any],
    save_dir: str | Path | None = None,
) -> dict[str, float]:
    model.eval()
    meter = BinarySegmentationMeter(
        threshold=float(config["threshold"]),
        compute_hd95=bool(config["compute_hd95"]),
    )
    loss_totals = {"segmentation_loss": 0.0, "boundary_loss": 0.0, "total_loss": 0.0}
    loss_steps = 0
    saved_count = 0
    paper_panel_paths: list[str] = []

    progress = tqdm(loader, desc="predict", leave=False)
    for batch in progress:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        boundaries = batch["boundary"].to(device, non_blocking=True)
        output = model(images)
        loss, loss_parts = total_loss(output, masks, boundaries, float(config["boundary_loss_weight"]))
        probs = (torch.sigmoid(output["logits"]) >= float(config["threshold"])).float()
        meter.update(output["logits"], masks, float(loss.item()))
        for key in loss_totals:
            loss_totals[key] += float(loss_parts[key])
        loss_steps += 1

        if save_dir is not None and saved_count < int(config["max_visualizations"]):
            batch_size = images.shape[0]
            for idx in range(batch_size):
                if saved_count >= int(config["max_visualizations"]):
                    break
                saved_paths = save_prediction_triplet(
                    save_dir=save_dir,
                    sample_id=str(batch["sample_id"][idx]),
                    image=batch["image"][idx].cpu(),
                    target_mask=batch["mask"][idx].cpu(),
                    pred_mask=probs[idx].cpu(),
                    mean=config["normalize_mean"],
                    std=config["normalize_std"],
                )
                paper_panel_paths.append(saved_paths["paper_panel"])
                saved_count += 1

    if save_dir is not None and paper_panel_paths:
        save_contact_sheet(
            panel_paths=paper_panel_paths,
            output_path=Path(save_dir) / "paper_contact_sheet.png",
            columns=int(config["contact_sheet_columns"]),
        )

    metrics = meter.compute()
    metrics.update(mean_scalar_dict(loss_totals, loss_steps))
    return metrics


def plot_training_history(history_df: pd.DataFrame) -> None:
    if history_df.empty:
        print("No training history to plot yet.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history_df["epoch"], history_df["train_total_loss"], label="train")
    axes[0].plot(history_df["epoch"], history_df["val_total_loss"], label="val")
    axes[0].set_title("Total Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["train_dice"], label="train")
    axes[1].plot(history_df["epoch"], history_df["val_dice"], label="val")
    axes[1].set_title("Dice")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


## Build Model and Optimizer

This cell instantiates the model, optionally loads ImageNet weights into the encoder, and runs one forward-pass sanity check.


In [ ]:
MODEL = WBSNet(CONFIG).to(DEVICE)
if CONFIG["encoder_pretrained"]:
    MODEL.encoder.load_imagenet_pretrained(fail_silently=bool(CONFIG["allow_pretrained_fallback"]))

OPTIMIZER, SCHEDULER = build_optimizer(MODEL, CONFIG)
SCALER = GradScaler(enabled=bool(CONFIG["amp"] and DEVICE.type == "cuda"))

total_params, trainable_params = count_parameters(MODEL)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

example_batch = next(iter(VAL_LOADER))
with torch.no_grad():
    example_output = MODEL(example_batch["image"][:2].to(DEVICE))

print(f"Input batch shape:    {tuple(example_batch['image'][:2].shape)}")
print(f"Segmentation logits:  {tuple(example_output['logits'].shape)}")
print(f"Boundary heads:       {[tuple(item.shape) for item in example_output['boundary_logits']]}")


## Training Loop

Run the next cell to train the model. It will:
- track train and validation metrics every epoch
- save `best.pt` based on the monitored validation metric
- save `last.pt` at the end
- write `metrics.csv` and `best_metrics.json` inside the run directory


In [ ]:
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "best.pt"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last.pt"

history: list[dict[str, Any]] = []
start_epoch = 0
best_metric = float("-inf") if CONFIG["monitor_mode"] == "max" else float("inf")
best_metrics_snapshot: dict[str, float] = {}

resume_checkpoint = CONFIG.get("resume_checkpoint")
if resume_checkpoint:
    checkpoint = load_checkpoint(
        resume_checkpoint,
        MODEL,
        optimizer=OPTIMIZER,
        scheduler=SCHEDULER,
        scaler=SCALER,
        map_location=DEVICE,
    )
    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_metric = float(checkpoint.get("best_metric", best_metric))
    if METRICS_CSV.exists():
        history = pd.read_csv(METRICS_CSV).to_dict("records")
    print(f"Resumed from {resume_checkpoint} at epoch {start_epoch}.")

for epoch in range(start_epoch, int(CONFIG["epochs"])):
    train_metrics = run_epoch(
        model=MODEL,
        loader=TRAIN_LOADER,
        optimizer=OPTIMIZER,
        scaler=SCALER,
        device=DEVICE,
        config=CONFIG,
        training=True,
    )
    val_metrics = run_epoch(
        model=MODEL,
        loader=VAL_LOADER,
        optimizer=None,
        scaler=SCALER,
        device=DEVICE,
        config=CONFIG,
        training=False,
    )
    SCHEDULER.step()

    row = {
        "epoch": epoch + 1,
        **{f"train_{key}": value for key, value in train_metrics.items()},
        **{f"val_{key}": value for key, value in val_metrics.items()},
        "encoder_lr": OPTIMIZER.param_groups[0]["lr"],
        "decoder_lr": OPTIMIZER.param_groups[1]["lr"],
    }
    history.append(row)
    history_df = pd.DataFrame(history)
    history_df.to_csv(METRICS_CSV, index=False)

    current_metric = float(val_metrics[CONFIG["monitor"]])
    improved = current_metric > best_metric if CONFIG["monitor_mode"] == "max" else current_metric < best_metric
    if improved:
        best_metric = current_metric
        best_metrics_snapshot = dict(val_metrics)
        save_checkpoint(
            BEST_CHECKPOINT_PATH,
            MODEL,
            OPTIMIZER,
            SCHEDULER,
            SCALER,
            epoch=epoch,
            best_metric=best_metric,
            config=CONFIG,
        )
        save_json(RUN_DIR / "best_metrics.json", best_metrics_snapshot)

    if (epoch + 1) % int(CONFIG["save_every"]) == 0:
        save_checkpoint(
            CHECKPOINT_DIR / f"epoch_{epoch + 1:03d}.pt",
            MODEL,
            OPTIMIZER,
            SCHEDULER,
            SCALER,
            epoch=epoch,
            best_metric=best_metric,
            config=CONFIG,
        )

    print(
        f"Epoch {epoch + 1:03d}/{CONFIG['epochs']:03d} | "
        f"train -> {format_metrics(train_metrics, ['total_loss', 'dice', 'iou', 'hd95'])} | "
        f"val -> {format_metrics(val_metrics, ['total_loss', 'dice', 'iou', 'hd95'])}"
    )

save_checkpoint(
    LAST_CHECKPOINT_PATH,
    MODEL,
    OPTIMIZER,
    SCHEDULER,
    SCALER,
    epoch=max(int(CONFIG["epochs"]) - 1, 0),
    best_metric=best_metric,
    config=CONFIG,
)

if not best_metrics_snapshot and BEST_CHECKPOINT_PATH.exists():
    best_metrics_snapshot = json.loads((RUN_DIR / "best_metrics.json").read_text(encoding="utf-8"))

history_df = pd.DataFrame(history)
display(history_df.tail())
plot_training_history(history_df)
print(f"Best {CONFIG['monitor']}: {best_metric:.4f}")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")


## Final Evaluation and Prediction Export

This cell reloads the best checkpoint, evaluates the configured splits, writes JSON summaries, and saves qualitative prediction panels plus contact sheets.


In [ ]:
if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError("Best checkpoint not found. Run the training cell first.")

best_checkpoint = load_checkpoint(BEST_CHECKPOINT_PATH, MODEL, map_location=DEVICE)
print(f"Loaded best checkpoint from epoch {best_checkpoint.get('epoch', 'unknown')}")
print(f"Best monitored metric: {best_checkpoint.get('best_metric', float('nan')):.4f}")

evaluation_rows: list[dict[str, Any]] = []
for spec in EVAL_SPECS:
    dataset_name = spec["dataset"]
    split_name = spec["split"]
    eval_dataset = ProcessedWBSNetDataset(
        processed_root=PROCESSED_ROOT,
        dataset_name=dataset_name,
        split=split_name,
        train=False,
        config=CONFIG,
    )
    eval_loader = build_dataloader(
        eval_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        config=CONFIG,
    )
    export_dir = EVAL_DIR / f"{dataset_name}_{split_name}"
    metrics = evaluate_and_export(
        model=MODEL,
        loader=eval_loader,
        device=DEVICE,
        config=CONFIG,
        save_dir=export_dir,
    )
    payload = {
        "dataset": dataset_name,
        "split": split_name,
        "metrics": metrics,
        "checkpoint": str(BEST_CHECKPOINT_PATH),
        "run_name": RUN_NAME,
        "processed_root": str(PROCESSED_ROOT),
    }
    save_json(export_dir / "metrics.json", payload)
    evaluation_rows.append({"dataset": dataset_name, "split": split_name, **metrics})
    print(f"{dataset_name}/{split_name} -> {format_metrics(metrics, ['total_loss', 'dice', 'iou', 'hd95'])}")

evaluation_df = pd.DataFrame(evaluation_rows).sort_values(["dataset", "split"]).reset_index(drop=True)
save_json(EVAL_DIR / "evaluation_summary.json", evaluation_rows)
display(evaluation_df)

history_df = pd.read_csv(METRICS_CSV) if METRICS_CSV.exists() else pd.DataFrame(history)
plot_training_history(history_df)

contact_sheets = sorted(EVAL_DIR.glob("*/paper_contact_sheet.png"))
for contact_sheet_path in contact_sheets:
    print(contact_sheet_path)
    with Image.open(contact_sheet_path) as contact_sheet_image:
        display(contact_sheet_image.copy())
